# NIFTY-ARIMA

Forecasting 1-minute NIFTY 50 prices (January 2021) with ARIMA, benchmarked against a random-walk
(persistence) baseline.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apoorvsingh090/NIFTY-ARIMA/blob/main/TIMESERIES_NIFTY.ipynb)

## Overview

The notebook evaluates several one-step-ahead forecasters on the same series:

| Model | Idea |
| --- | --- |
| Persistence | tomorrow's price = last observed price (the baseline to beat) |
| Moving average | mean of the last *w* prices |
| AR (`AutoReg`) | regress price on its own lags |
| MA (residual AR) | regress the *persistence error* on its own lags and correct |
| ARIMA | classical ARIMA, order selected by AIC on the training set only |

Everything runs through `nifty_arima.py` so the logic is testable outside the notebook. All models
use one shared train/test split (`na.TRAIN_FRACTION`) and forecasts are snapped to the 0.05 tick.

### How to run

1. `pip install -r requirements.txt`
2. Download the *Intraday 1 Min Data* dataset and point the loader at
   `Intraday 1 Min Data/2021/2021 JAN NIFTY.txt` (see the next cell).
3. Run the notebook top to bottom.

In Colab, run the bootstrap cell first; it clones the repo and installs dependencies.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/apoorvsingh090/NIFTY-ARIMA.git"

if not os.path.exists("nifty_arima.py"):
    subprocess.run(["git", "clone", "-q", REPO_URL, ".repo"], check=True)
    sys.path.insert(0, ".repo")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", ".repo/requirements.txt"],
        check=True,
    )

import matplotlib.pyplot as plt
import pandas as pd

import nifty_arima as na

print(f"train fraction={na.TRAIN_FRACTION}, tick={na.TICK}")

## Load data

The source file is headerless: a serial column, then date/time/OHLC, then two unused columns.
`load_intraday` drops the extras and builds a real `DatetimeIndex`, so session gaps are explicit
rather than hidden behind a positional index.

In [ ]:
# Colab: mount Drive and point at the Kaggle extract instead, e.g.
#   from google.colab import drive; drive.mount("/content/drive")
#   DATA_PATH = "/content/drive/MyDrive/kaggle/Intraday 1 Min Data/2021/2021 JAN NIFTY.txt"
DATA_PATH = os.environ.get("NIFTY_DATA", "Intraday 1 Min Data/2021/2021 JAN NIFTY.txt")

df = na.load_intraday(DATA_PATH)
close = df["close"]

print(df.shape)
print(df.index.min(), "->", df.index.max())
df.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8))
close.plot(ax=axes[0])
axes[0].set_title("NIFTY close (1-minute)")
close.hist(ax=axes[1], bins=60)
axes[1].set_title("distribution of closes")
close.plot(kind="kde", ax=axes[2])
axes[2].set_title("density")
fig.tight_layout()

## Is the series stationary?

ARIMA's `d` term is only justified if the series needs differencing, so test the level first and then
the first difference with ADF.

In [ ]:
na.adf_summary(close, "close level")

In [ ]:
na.adf_summary(close.diff().dropna(), "first difference")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

diff = close.diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
plot_acf(diff, lags=120, ax=axes[0])
plot_pacf(diff, lags=40, ax=axes[1], method="ywm")
fig.tight_layout()

## Baselines

A forecasting result is only meaningful next to a naive baseline, so persistence and a moving average
are computed first.

In [ ]:
persistence = na.persistence_forecast(close)
persistence.plot()

In [ ]:
moving_average = na.moving_average_forecast(close, window=3)
moving_average.plot(zoom=300)

## Autoregressive models

`AutoReg` replaces the removed `statsmodels.tsa.ar_model.AR`; the lag order is chosen by AIC on the
training slice. The second model applies AR to the persistence residuals (a moving-average style
correction).

In [ ]:
autoregression = na.walk_forward_ar(close)
autoregression.plot(zoom=300)

In [ ]:
residual_ar = na.residual_correction_forecast(close)
residual_ar.plot(zoom=300)

## ARIMA

Order selection uses AIC on **training data only**; the test period is not touched during tuning, so the
reported error is honest. `walk_forward_arima` forecasts one step ahead, appends the observed value, and
refits — no future information leaks in.

In [ ]:
orders = [(p, 1, q) for p in (0, 1, 2, 4) for q in (0, 1, 2)]
n_train = int(len(close) * na.TRAIN_FRACTION)
train_close = close.iloc[:n_train]

aic_table = na.select_order_by_aic(train_close, orders)
best_order = tuple(aic_table.index[0])
print(f"selected on training data only: ARIMA{best_order}")
aic_table.head(8).round(2)

In [ ]:
arima = na.walk_forward_arima(close, best_order)

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
arima.plot(ax=axes[0])
arima.plot(zoom=300, ax=axes[1])
axes[1].set_title(f"{arima.name} - first 300 test steps")
fig.tight_layout()

## Results

All models share the same split, so RMSE/MAE/sMAPE/MASE are directly comparable.
`dir_acc` is the share of 1-minute moves whose direction was predicted correctly (0.5 = coin flip);
`ci95_coverage` is how often the actual value landed inside ARIMA's 95% interval — ideally ~0.95.

In [ ]:
results = {
    "Persistence": persistence,
    "Moving average (w=3)": moving_average,
    "AR (AutoReg)": autoregression,
    "MA (residual AR)": residual_ar,
    arima.name: arima,
}

table = pd.DataFrame([r.summary_row() for r in results.values()]).set_index("model")
baseline_rmse = table.loc["Persistence", "rmse"]
table["rmse_gain_pct"] = (baseline_rmse - table["rmse"]) / baseline_rmse * 100

table.sort_values("rmse").round(3)

## Residual diagnostics

If a model has captured the structure, its residuals should look like white noise: roughly normal,
no strong autocorrelation, and a Ljung-Box p-value above 0.05.

In [ ]:
na.plot_residual_diagnostics(arima)
plt.show()

na.ljung_box_test(arima.residuals, lags=(10, 20))

## What the numbers mean

- **Compare against persistence first.** If ARIMA's RMSE is not below the persistence RMSE, the extra
  parameters are not earning their keep. On 1-minute equity prices that is the usual outcome: the series
  is close to a random walk.
- **MASE < 1** beats the naive forecast; **MASE > 1** is worse than guessing "same as last time".
- **`dir_acc` near 0.5** means the model has no directional edge, which is expected at 1-minute
  resolution and is a fair conclusion to report, not a failure to hide.
- **`ci95_coverage` should sit near 0.95.** Wide intervals that are rarely violated, or narrow ones that
  are frequently violated, mean the uncertainty estimate is miscalibrated.
- **Ljung-Box p-values > 0.05** mean no significant autocorrelation is left in the residuals.

### Known limitations

- 1-minute data is not equally spaced: overnight gaps, weekends and holidays are absent. ARIMA treats the
  index as regular, so calendar effects are not modeled.
- Only one month of data is used, so results should not be generalized.
- No transaction costs or volatility modeling; this is a forecasting exercise, not a trading strategy.